In [2]:
import os
import zlib
import stat
import subprocess
import time
import re

def run_command(command):
    """Runs a shell command and returns the result object."""
    print(f"\nRunning: {' '.join(command)}")
    result = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    if result.stdout:
        print("Output:")
        print(result.stdout)
    if result.stderr:
        print("Errors:")
        print(result.stderr)
    return result

def inspect_git_objects(repo_objects_path=".git/objects"):
    """
    Iterates through all loose objects, attempts to decompress them,
    and collects the hashes of any that fail.
    """
    corrupt_objects = []
    total_objects = 0

    for subdir in os.listdir(repo_objects_path):
        subdir_path = os.path.join(repo_objects_path, subdir)
        if not os.path.isdir(subdir_path) or subdir in ["info", "pack"]:
            continue

        for filename in os.listdir(subdir_path):
            total_objects += 1
            obj_hash = subdir + filename
            object_path = os.path.join(subdir_path, filename)
            try:
                with open(object_path, "rb") as f:
                    data = f.read()
                # Attempt to decompress the object data
                zlib.decompress(data)
            except Exception as e:
                print(f"Error in object {obj_hash} at {object_path}: {e}")
                corrupt_objects.append(obj_hash)

    print("\nSummary of loose objects inspection:")
    print(f"Total objects inspected: {total_objects}")
    print(f"Total corrupt objects: {len(corrupt_objects)}")
    if corrupt_objects:
        print("List of corrupt object hashes:")
        for h in corrupt_objects:
            print(h)
    else:
        print("No corrupt objects found.")

    return corrupt_objects

def object_is_referenced(obj_hash):
    """
    Check if an object is referenced by any commit in the repository.
    Uses 'git rev-list --all --objects' and checks if the object hash appears in the output.
    """
    result = run_command(["git", "rev-list", "--all", "--objects"])
    return obj_hash in result.stdout

def remove_corrupt_objects(corrupt_objects, repo_objects_path=".git/objects"):
    """
    Removes each corrupt object file. Even if an object appears to be referenced,
    it will be removed (with a warning) to clean the repository.
    """
    for obj_hash in corrupt_objects:
        obj_dir = os.path.join(repo_objects_path, obj_hash[:2])
        obj_file = os.path.join(obj_dir, obj_hash[2:])
        
        if object_is_referenced(obj_hash):
            print(f"WARNING: Object {obj_hash} is referenced in history, but it will be removed to clean corruption.")

        try:
            os.remove(obj_file)
            print(f"Removed corrupt object {obj_hash} at {obj_file}")
        except PermissionError as e:
            print(f"PermissionError when removing {obj_hash} at {obj_file}: {e}")
            try:
                os.chmod(obj_file, stat.S_IWRITE)
                os.remove(obj_file)
                print(f"Removed corrupt object {obj_hash} at {obj_file} after changing permissions")
            except Exception as inner_e:
                print(f"Failed to remove {obj_hash} at {obj_file} even after chmod: {inner_e}")
                print("Please run the script as administrator or check file attributes manually.")
        except Exception as e:
            print(f"Failed to remove object {obj_hash} at {obj_file}: {e}")

def check_fsck():
    """Run git fsck --full to check repository integrity."""
    return run_command(["git", "fsck", "--full"])

def parse_fsck_missing_blobs(fsck_output):
    """
    Parse fsck output for missing blob errors.
    Returns a list of missing blob hashes.
    """
    missing_blobs = []
    # Lines usually look like: "missing blob <hash>"
    for line in fsck_output.splitlines():
        m = re.search(r"missing blob ([0-9a-f]+)", line)
        if m:
            missing_blobs.append(m.group(1))
    return missing_blobs

def run_gc():
    """
    Run garbage collection and repacking. If GC is already running, wait and retry.
    """
    # Expire reflogs first
    run_command(["git", "reflog", "expire", "--expire=now", "--all"])

    # Now run garbage collection.
    result = run_command(["git", "gc", "--prune=now", "--aggressive", "--force"])
    if "gc is already running" in result.stderr:
        print("GC is already running. Waiting 5 seconds before retrying...")
        time.sleep(5)
        result = run_command(["git", "gc", "--prune=now", "--aggressive", "--force"])
    return result

def verify_packfiles():
    """
    Inspect packfiles using git verify-pack -v.
    If a packfile appears corrupted (non-zero return or errors in output),
    it will be removed.
    """
    pack_dir = ".git/objects/pack"
    if os.path.isdir(pack_dir):
        for filename in os.listdir(pack_dir):
            if filename.endswith(".idx"):
                pack_idx_path = os.path.join(pack_dir, filename)
                result = run_command(["git", "verify-pack", "-v", pack_idx_path])
                if result.returncode != 0 or "error" in result.stderr.lower():
                    pack_file = pack_idx_path.replace(".idx", ".pack")
                    print(f"Removing corrupt pack file: {pack_file} and index {pack_idx_path}")
                    try:
                        os.remove(pack_file)
                        os.remove(pack_idx_path)
                    except Exception as e:
                        print(f"Failed to remove pack files {pack_file} and {pack_idx_path}: {e}")
    else:
        print("No pack directory found; skipping packfile verification.")

def remove_packfiles_with_missing_blobs(missing_blobs):
    """
    For each packfile in .git/objects/pack, check if its verify-pack output mentions
    any missing blob hash. If so, remove both the .pack and .idx files.
    """
    pack_dir = ".git/objects/pack"
    if os.path.isdir(pack_dir):
        for idx_file in os.listdir(pack_dir):
            if idx_file.endswith(".idx"):
                pack_idx_path = os.path.join(pack_dir, idx_file)
                result = run_command(["git", "verify-pack", "-v", pack_idx_path])
                # Check if any missing blob hash appears in the verify-pack output.
                should_remove = any(blob in result.stdout or blob in result.stderr for blob in missing_blobs)
                if should_remove:
                    pack_file = pack_idx_path.replace(".idx", ".pack")
                    print(f"Removing pack file {pack_file} and index {pack_idx_path} because they reference missing blobs.")
                    try:
                        os.remove(pack_file)
                        os.remove(pack_idx_path)
                    except Exception as e:
                        print(f"Failed to remove pack file pair {pack_file} and {pack_idx_path}: {e}")
    else:
        print("No pack directory found; skipping missing blob packfile removal.")

def check_git_status():
    """Check the git status to ensure the working directory is clean."""
    run_command(["git", "status"])

def fetch_all_objects():
    """Fetch all objects from the remote to restore any missing objects."""
    print("\n== Fetching all objects from remote ==")
    run_command(["git", "fetch", "--all", "--prune"])

def main_loop():
    """
    Repeatedly inspects and removes corrupt loose objects until none remain.
    Then runs integrity checks.
    """
    iteration = 1
    while True:
        print(f"\n--- Inspection iteration {iteration} ---")
        corrupt_objects = inspect_git_objects()
        if not corrupt_objects:
            print("No corrupt loose objects remain.")
            break
        # Automatically remove detected corrupt objects.
        remove_corrupt_objects(corrupt_objects)
        print("\nRunning 'git fsck --full' to check repository integrity...")
        fsck_result = check_fsck()
        missing_blobs = parse_fsck_missing_blobs(fsck_result.stdout + fsck_result.stderr)
        if missing_blobs:
            print("\nDetected missing blobs:")
            for blob in missing_blobs:
                print(blob)
        else:
            print("\nNo missing blobs reported by fsck.")
        # Pause briefly before the next iteration.
        time.sleep(1)
        iteration += 1

if __name__ == "__main__":
    print("== Starting Git Repository Integrity Script ==")
    
    # First, inspect and remove any corrupt loose objects.
    main_loop()

    # Run an integrity check.
    print("\n== Running additional Git integrity checks ==")
    fsck_result = check_fsck()
    missing_blobs = parse_fsck_missing_blobs(fsck_result.stdout + fsck_result.stderr)
    if missing_blobs:
        print("\nFinal fsck reports missing blobs:")
        for blob in missing_blobs:
            print(blob)
    else:
        print("\nNo missing blobs reported by final fsck.")

    # Run garbage collection.
    print("\n== Running Git garbage collection ==")
    run_gc()

    # Verify packfiles and remove any that are outright broken.
    print("\n== Verifying packfiles ==")
    verify_packfiles()

    # If missing blobs remain, remove any packfiles referencing them.
    if missing_blobs:
        print("\n== Removing packfiles referencing missing blobs ==")
        remove_packfiles_with_missing_blobs(missing_blobs)
        print("\n== Running Git garbage collection again after packfile removal ==")
        run_gc()
    
    print("\n== Checking Git status ==")
    check_git_status()

    print("\n== Fetching all objects from remote (to restore missing ones) ==")
    fetch_all_objects()

    print("\n== Final repository integrity check ==")
    fsck_result = check_fsck()
    missing_blobs = parse_fsck_missing_blobs(fsck_result.stdout + fsck_result.stderr)
    if missing_blobs:
        print("\nFinal fsck still reports missing blobs:")
        for blob in missing_blobs:
            print(blob)
    else:
        print("\nNo missing blobs reported by final fsck.")

    print("\nAll checks complete. Your repository should now be in a healthier state for pushing to origin.")


== Starting Git Repository Integrity Script ==

--- Inspection iteration 1 ---

Summary of loose objects inspection:
Total objects inspected: 9210
Total corrupt objects: 0
No corrupt objects found.
No corrupt loose objects remain.

== Running additional Git integrity checks ==

Running: git fsck --full
Output:
missing blob 4584cb639e05c31fbf84a9054a002c3c23b2e07a
missing blob dc8af4f068ee3ef55ab7b75f4fff2e475d14ad66
missing blob 4d15bb155fa81da9a1c5733c8a492938dfc00f86
missing blob 709840401cfccf7914ff08006ed2ccda83717b1e
missing blob 49996f48de572f8a640dd32aeffe8fe77af2f186
missing blob 3a2a85c637d4a18aca7d14fde1e75105f53fb495
missing blob 80b2199bef98f9734ca99efa134983d6bbb6b5d1
missing blob 8ebd0f58a66a349805ce06fe25c5d16504e61c86
missing blob ba4a1c5dd08bcfbc722d95eb70743a276cad257d
missing blob 474f1fda9abc051fe40a9c77be499d94103483ba
missing blob 85d75aaa15458dfcdceb4bc2b11cb4dbb88a45c3
missing blob 0ee172644eba6981b019cd206a79d5de513b8ec1
missing blob 1661fafc5099f7c0d7cb8d59121